# Regular Languages

This notebook ties together the two previous topics. **Regular expressions** (notebook 03) and **finite state machines** (notebook 13) turn out to describe the *same* collection of languages — the **regular languages**. We start with the vocabulary of formal languages, then see the operations that build them, and finish with the famous limit on what regular languages can express.

## Alphabets, Strings, and Languages

- An **alphabet** $\Sigma$ is a finite set of **symbols**, e.g. $\Sigma = \{0, 1\}$ or $\Sigma = \{a, b, c\}$.
- A **string** (or word) over $\Sigma$ is a finite sequence of symbols. The **empty string** is written $\varepsilon$ (length $0$).
- The **length** of a string $w$ is written $|w|$.
- $\Sigma^{*}$ is the set of **all** strings over $\Sigma$ (including $\varepsilon$). It is infinite.
- A **language** $L$ over $\Sigma$ is any subset $L \subseteq \Sigma^{*}$ — a (possibly infinite) set of strings.

For example, over $\Sigma = \{0,1\}$ the set of strings with an even number of $1$s is a language.

In [1]:
from itertools import product


def sigma_star(alphabet, max_len):
    """All strings over `alphabet` up to length max_len (including the empty string)."""
    words = [""]
    for length in range(1, max_len + 1):
        for tup in product(sorted(alphabet), repeat=length):
            words.append("".join(tup))
    return words


Sigma = {"0", "1"}
words = sigma_star(Sigma, 3)
print(f"Sigma* over {{0,1}} up to length 3  ({len(words)} strings):")
print(["(empty)" if w == "" else w for w in words])

# A language is just the subset satisfying some property.
even_ones = [w for w in words if w.count("1") % 2 == 0]
print("\nLanguage 'even number of 1s' (length <= 3):", ["(empty)" if w == "" else w for w in even_ones])

Sigma* over {0,1} up to length 3  (15 strings):
['(empty)', '0', '1', '00', '01', '10', '11', '000', '001', '010', '011', '100', '101', '110', '111']

Language 'even number of 1s' (length <= 3): ['(empty)', '0', '00', '11', '000', '011', '101', '110']


## Operations on Languages

New languages are built from old ones with three core operations. Let $L_1, L_2 \subseteq \Sigma^{*}$.

- **Union:** $L_1 \cup L_2$ — strings in either language.
- **Concatenation:** $L_1 L_2 = \{xy : x \in L_1,\ y \in L_2\}$ — an $L_1$ string followed by an $L_2$ string.
- **Kleene star:** $L^{*} = \{\varepsilon\} \cup L \cup LL \cup LLL \cup \cdots$ — zero or more strings from $L$ concatenated together.

These three operations are exactly what regular expressions express (`|`, juxtaposition, and `*`).

In [2]:
def concat(L1, L2):
    return {x + y for x in L1 for y in L2}


def kleene_star(L, max_len):
    """All strings obtainable by concatenating words of L, truncated to length max_len."""
    L = {w for w in L if w}            # drop empty word to keep the loop finite
    result = {""}
    frontier = {""}
    while frontier:
        nxt = set()
        for w in frontier:
            for x in L:
                z = w + x
                if len(z) <= max_len and z not in result:
                    nxt.add(z)
        result |= nxt
        frontier = nxt
    return result


A = {"a", "b"}
B = {"c"}
print("A union B    :", sorted(A | B))
print("A concat B   :", sorted(concat(A, B)))
print("{ab}* (len<=6):", sorted(kleene_star({"ab"}, 6)))

A union B    : ['a', 'b', 'c']
A concat B   : ['ac', 'bc']
{ab}* (len<=6): ['', 'ab', 'abab', 'ababab']


## Regular Languages and Kleene's Theorem

A language is **regular** if it can be built from the empty language, $\{\varepsilon\}$, and single symbols using finitely many applications of **union, concatenation, and Kleene star** — in other words, if some **regular expression** describes it.

**Kleene's Theorem.** A language is regular **if and only if** some finite automaton recognizes it.

So three views describe the same class of languages:

$$\textbf{regular expression} \;\Longleftrightarrow\; \textbf{regular language} \;\Longleftrightarrow\; \textbf{finite automaton}.$$

Let us *demonstrate* this equivalence for the "even number of 1s" language: a DFA and a regular expression should accept exactly the same strings.

In [3]:
import re

# (1) DFA recognizer (the machine from notebook 13).
def even_ones_dfa(s):
    state = "even"
    for ch in s:
        state = {"even": {"0": "even", "1": "odd"},
                 "odd":  {"0": "odd",  "1": "even"}}[state][ch]
    return state == "even"

# (2) Regular-expression recognizer for the same language.
even_ones_re = re.compile(r"^0*(10*10*)*$")

# Compare the two recognizers on every string up to length 8.
words = sigma_star({"0", "1"}, 8)
agree = all(even_ones_dfa(w) == bool(even_ones_re.match(w)) for w in words)
print(f"Checked {len(words)} strings (length <= 8).")
print("DFA and regex agree on every string:", agree)
print("\nExamples:")
for w in ["", "11", "1", "0110", "101"]:
    print(f"  {w or '(empty)':>8}: DFA={even_ones_dfa(w)!s:<5} regex={bool(even_ones_re.match(w))}")

Checked 511 strings (length <= 8).
DFA and regex agree on every string: True

Examples:
   (empty): DFA=True  regex=True
        11: DFA=True  regex=True
         1: DFA=False regex=False
      0110: DFA=True  regex=True
       101: DFA=True  regex=True


## Closure Properties

The regular languages are **closed** under many operations: if $L_1$ and $L_2$ are regular, then so are

$$L_1 \cup L_2,\quad L_1 L_2,\quad L_1^{*},\quad \overline{L_1}\ (\text{complement}),\quad L_1 \cap L_2.$$

This is enormously useful: you can combine simple regular pieces and stay within the regular world, guaranteed to still be recognizable by a finite automaton.

## A Limit: The Pumping Lemma

Not every language is regular. The classic counterexample is

$$L = \{\, 0^n 1^n : n \ge 0 \,\} = \{\varepsilon,\ 01,\ 0011,\ 000111,\ \dots\}.$$

Matching $0^n 1^n$ requires *counting* the $0$s so you can compare against the $1$s — but a finite automaton has only finitely many states and cannot remember an unbounded count. The **pumping lemma** makes this precise: every regular language has a length $p$ such that any string at least that long can be "pumped" (a middle part repeated) and stay in the language. For $0^n1^n$, pumping the middle breaks the equal-count condition, so $L$ is **not regular**.

The recognizer below works only because Python integers can count without limit — it is *not* a finite-state device.

In [4]:
def in_0n1n(s):
    # Recognize 0^n 1^n: some run of 0s followed by an equal run of 1s.
    i = 0
    while i < len(s) and s[i] == "0":
        i += 1
    zeros = i
    while i < len(s) and s[i] == "1":
        i += 1
    return i == len(s) and zeros == len(s) - zeros

for s in ["", "01", "0011", "000111", "001", "0101", "10"]:
    print(f"{s or '(empty)':>8}: in 0^n1^n? {in_0n1n(s)}")

print("\nNo finite automaton can recognize this language — it would need to count without bound.")

 (empty): in 0^n1^n? True
      01: in 0^n1^n? True
    0011: in 0^n1^n? True
  000111: in 0^n1^n? True
     001: in 0^n1^n? False
    0101: in 0^n1^n? False
      10: in 0^n1^n? False

No finite automaton can recognize this language — it would need to count without bound.


## Worked Examples

A few examples solved with Python. (The exercises in the next section are for you to solve — their solutions live in the matching notebook in `solutions/`.)

### Example 1: Strings of a Given Length

List every string of length exactly 2 over the alphabet $\{0, 1\}$.

In [5]:
from itertools import product

print(["".join(t) for t in product("01", repeat=2)])

['00', '01', '10', '11']


### Example 2: Membership in a*b*

The language $a^* b^*$ contains strings of zero or more $a$'s followed by zero or more $b$'s. Recognize it with a regular expression.

In [6]:
import re

pattern = re.compile(r"^a*b*$")
for s in ["", "aaabb", "ab", "ba", "aabbb"]:
    print(f"{s or '(empty)':>8}: in a*b*? {bool(pattern.match(s))}")

 (empty): in a*b*? True
   aaabb: in a*b*? True
      ab: in a*b*? True
      ba: in a*b*? False
   aabbb: in a*b*? True


## Regular Languages Practice Problems

Write your Python in the code cell beneath each problem (some starter code is provided). Worked solutions are in [`solutions/14_regular_languages_key.ipynb`](solutions/14_regular_languages_key.ipynb).

### Problem 1: Generate Sigma*

For the alphabet $\Sigma = \{a\}$, generate every string in $\Sigma^{*}$ up to length 3 (including the empty string).

In [7]:
from itertools import product
# WRITE YOUR CODE BELOW


### Problem 2: Concatenation of Languages

Compute the concatenation $L_1 L_2$ where $L_1 = \{a, b\}$ and $L_2 = \{c, d\}$.

In [8]:
L1 = {"a", "b"}
L2 = {"c", "d"}
# WRITE YOUR CODE BELOW


### Problem 3: Recognize a Regular Language

Write a regular expression that recognizes binary strings with an **even** number of 1s, and test it on several strings.

In [9]:
import re
tests = ["", "11", "1", "1010", "111"]
# WRITE YOUR CODE BELOW


### Problem 4: A Non-Regular Language

The language $\{0^n 1^n : n \ge 0\}$ is **not** regular (a finite automaton cannot count without bound). Write a recognizer for it using a counter, and test it on several strings.

In [10]:
tests = ["", "01", "0011", "001", "10"]
# WRITE YOUR CODE BELOW
